In [9]:
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
import numpy as np

In [10]:
df_raw = pd.read_csv

In [ ]:
##Step 2: Find Your Weather Station
##Query the stations table to find every US station whose name contains 'LA GUARDIA', and record its usaf ID. You will use this ID in Step 7.

SELECT usaf
FROM stations
WHERE name LIKE '%LA GUARDIA%';

In [ ]:
##Step 3: Preview the Ride Data
##Before analyzing anything, look at what you're working with. Pull the starttime, stoptime, and tripduration for 10 trips from the citibike_trips table.
 ## Tip: Never run SELECT * without LIMIT on a table this size. Previewing first is not just politeness — the sandbox gives you a monthly query quota, and careless full-table scans burn through it.

SELECT starttime, stoptime, tripduration
FROM citibike_trips
LIMIT 10;

In [ ]:
##Step 4: Size Up the Problem
##How many trips are in the table, total? Write a query that returns a single number.
##In your submission document, answer in one sentence: why can't we just export this table to a CSV and open it in pandas?
 ##Tip: One aggregate function, no grouping needed.

 SELECT COUNT(*)
 FROM trips;


In [ ]:
##Step 5: Rides Per Day
##This is the heart of the assignment. Collapse the trip-level data into one row per day: for each calendar date, count the number of rides. Name the date column ride_date and the count num_rides. Exclude the junk rows where starttime is NULL.
##Checkpoint: Your dates should run from 2013-07-01 (the month Citi Bike launched) to 2018-05-31 (where this public table ends).
 ##Tip: starttime is a TIMESTAMP, but you want to group by calendar day. There is a function that extracts just the date part. And there is a specific SQL phrase for keeping only rows where a value is not absent.

SELECT
    DATE(starttime) AS ride_date
    COUNT(*) AS num_rides
FROM 
    citibike_trips
WHERE
    starttime IS NOT NULL
GROUP BY 
    DATE(starttime)
ORDER BY 
    ride_date ASC;

In [ ]:
## Step 6: Add Average Trip Duration
## Extend your Step 5 query: for each day, also compute the average trip length in MINUTES, named avg_duration_min.
 ## Tip: Check the schema reference for what units tripduration is stored in.

SELECT
    DATE(starttime) AS ride_date 
    ROUND(AVG(tripduration) / 60, 2) AS avg_duration_min
FROM 
    'bigquery-public-data.new_york_citibike.citibike_trips'
GROUP BY 
    1;



In [ ]:
## Step 7: One Station, Six Years of Weather
## Now the weather side. From the gsod20* wildcard tables, pull the daily observations for your LaGuardia station for 2013 through 2018. Your query should return: a proper DATE column named obs_date, plus temp, `max`, `min`, wdsp, and prcp — renamed to temp_f, max_temp_f, min_temp_f, wind_speed_knots, and precip_in.
## Two puzzles to solve here. First: year, mo, and da are three separate STRING columns, and you need one real DATE. Second: you must limit the wildcard to the right years.
## Checkpoint: Exactly 2,191 rows: one per day from 2013-01-01 to 2018-12-31. If you get more, your station filter is wrong; if fewer, your year range is.
 ## Tip: CONCAT the three string columns with '-' separators, then PARSE_DATE('%Y-%m-%d', ...) the result. Filter the wildcard with _TABLE_SUFFIX BETWEEN '13' AND '18'. And remember the backticks on `max` and `min` — they are reserved words.

SELECT
    PARSE_DATE('%Y-%m-%d', CONCAT(year, '-', mo, '-', da)) AS obs_date,
    temp AS temp_f,
    'max' AS max_temp_f,
    'min' AS min_temp_f,
    wdsp AS wind_speed_knots,
    prcp AS precip_in
FROM 
    `bigquery-public-data.noaa_gsod.gsod20*`
WHERE 
    _TABLE_SUFFIX BETWEEN '13' AND '18'
    AND stn = '25030'

In [ ]:
## Step 8: The Join — Your Final Query
## Assemble the final table. Using WITH, define your Step 6 query as a CTE named daily_rides and your Step 7 query as a CTE named daily_weather. Then INNER JOIN them on the date, and add two more columns computed from ride_date: day_of_week (the day's name, e.g. 'Monday') and month (a number 1–12). Order by ride_date.
## Final column list: ride_date, num_rides, avg_duration_min, temp_f, max_temp_f, min_temp_f, wind_speed_knots, precip_in, day_of_week, month.
## Checkpoint: Approximately 1,610 rows.
## In your submission document, answer: the weather CTE alone had 2,191 rows, but the joined table has ~1,610. Where did the other days go? (Think about what an INNER JOIN keeps, and which of the two sides is missing days.)
 ## Tip: FORMAT_DATE('%A', …) gives you the weekday name; EXTRACT(MONTH FROM …) gives you the month number.

WITH daily_rides AS (SELECT
    DATE(starttime) AS ride_date 
    ROUND(AVG(tripduration) / 60, 2) AS avg_duration_min
FROM 
    'bigquery-public-data.new_york_citibike.citibike_trips'
GROUP BY 
    1;),
daily weather AS (SELECT
    PARSE_DATE('%Y-%m-%d', CONCAT(year, '-', mo, '-', da)) AS obs_date,
    temp AS temp_f,
    'max' AS max_temp_f,
    'min' AS min_temp_f,
    wdsp AS wind_speed_knots,
    prcp AS precip_in
FROM 
    `bigquery-public-data.noaa_gsod.gsod20*`
WHERE 
    _TABLE_SUFFIX BETWEEN '13' AND '18'
    AND stn = '25030')
SELECT 
    dr.ride_date,
    dr.num_rides,
    dr.avg_duration_min,
    dw.min_temp_f,
    dw.max_temp_f,
    dw.min_temp_f,
    dw.wind_speed_knots,
    dw.precip_in,
    FORMAT_DATE('%A', dr.ride_date) AS day_of_week,
    EXTRACT(MONTH FROM dr.ride_date) AS month
FROM daily_rides AS dr 
INNER JOIN daily_weather AS dw 
    ON dr.ride_date = dw.date 
ORDER BY dr.ride_date


In [ ]:
## Step 9: Export and Commit
## Run your final query, then use Save Results → CSV (local file) in the BigQuery console. Name the file citibike_weather_daily.csv.
## Commit the CSV to the data/ folder of the repository you built in Step 1.
## Save your final Step 8 query as a file named build_dataset.sql and commit it to a queries/ folder.
## Push both to GitHub — you will build directly on this CSV in Part 2.
 ## Tip: Your future self is the customer here. If the CSV is wrong, Part 2 gets much harder — verify the checkpoint row count before you commit.
